# 01 · Exploración de maestros y facturación

**Proyecto:** Cloud Provider Analytics · **Instancia:** primera entrega (diseño y fundación) · **Fuentes:** 7 CSV de `datalake/landing/`

Perfil inicial de las fuentes batch: inventario, esquema, nulos, claves, rangos, consistencia interna e integridad referencial.
Los archivos de Landing **no se modifican**: este notebook solo los lee.

Los tratamientos que aparecen en el resumen final son **propuestas** para Bronze/Silver, no decisiones tomadas: los criterios de tratamiento se definen en la implementación.

**Cómo ejecutar**
Desde la raíz del repo con el venv activo, abrir el notebook (VS Code o `jupyter lab`) y ejecutar todas las celdas.

Este notebook usa pandas porque los maestros pesan ~0,3 MB. El procesamiento con PySpark comienza en la segunda entrega.

## 0 · Setup

Busca `datalake/landing/` subiendo desde la carpeta actual. Si el notebook corre fuera del repo, definí la variable de entorno `LANDING_DIR` con la ruta a los datos.

In [10]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 60)


def find_landing() -> Path:
    """Devuelve la carpeta datalake/landing (solo lectura: Landing es inmutable)."""
    env = os.environ.get("LANDING_DIR")
    if env and Path(env).is_dir():
        return Path(env)
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "datalake" / "landing"
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "No se encontró datalake/landing/. Ejecutá el notebook desde el repo o definí la variable "
        "de entorno LANDING_DIR con la ruta a los datos."
    )


LANDING = find_landing()
n_files = sum(1 for _ in LANDING.rglob("*") if _.is_file())
print(f"Landing encontrado ({n_files} archivos). pandas {pd.__version__}")

Landing encontrado (127 archivos). pandas 3.0.6


## 1 · Inventario de fuentes

Grano y clave candidata de cada fuente, y si la clave se repite.

In [11]:
SOURCES = {
    # nombre: (archivo, clave candidata, grano)
    "customers": ("customers_orgs.csv",    ["org_id"],                "Organización"),
    "users":     ("users.csv",             ["user_id"],               "Usuario"),
    "resources": ("resources.csv",         ["resource_id"],           "Recurso cloud"),
    "tickets":   ("support_tickets.csv",   ["ticket_id"],             "Ticket de soporte"),
    "marketing": ("marketing_touches.csv", ["touch_id"],              "Interacción de marketing"),
    "nps":       ("nps_surveys.csv",       ["org_id", "survey_date"], "Encuesta NPS por organización y fecha"),
    "billing":   ("billing_monthly.csv",   ["invoice_id"],            "Factura (organización, mes)"),
}

dfs = {k: pd.read_csv(LANDING / v[0]) for k, v in SOURCES.items()}

inventario = pd.DataFrame([
    {
        "fuente": k,
        "archivo": v[0],
        "filas": len(dfs[k]),
        "columnas": dfs[k].shape[1],
        "KB": round((LANDING / v[0]).stat().st_size / 1024, 1),
        "grano": v[2],
        "clave": " + ".join(v[1]),
        "claves repetidas": int(dfs[k].duplicated(v[1]).sum()),
    }
    for k, v in SOURCES.items()
])
inventario

,fuente,archivo,filas,columnas,KB,grano,clave,claves repetidas
0,customers,customers_orgs.csv,80,11,7.6,Organización,org_id,0
1,users,users.csv,800,7,69.0,Usuario,user_id,0
2,resources,resources.csv,400,7,35.8,Recurso cloud,resource_id,0
3,tickets,support_tickets.csv,1000,8,69.3,Ticket de soporte,ticket_id,0
4,marketing,marketing_touches.csv,1500,7,99.6,Interacción de marketing,touch_id,0
5,nps,nps_surveys.csv,92,4,3.9,Encuesta NPS por organización y fecha,org_id + survey_date,0
6,billing,billing_monthly.csv,240,8,15.5,"Factura (organización, mes)",invoice_id,0


## 2 · Esquema inferido

Tipos que infiere pandas al leer.

In [12]:
for k, d in dfs.items():
    print(f"{k:10s}", ", ".join(f"{c}:{t}" for c, t in d.dtypes.astype(str).items()))

customers  org_id:str, org_name:str, industry:str, hq_region:str, plan_tier:str, is_enterprise:bool, signup_date:str, sales_rep:str, lifecycle_stage:str, marketing_source:str, nps_score:float64
users      user_id:str, org_id:str, email:str, role:str, active:bool, created_at:str, last_login:str
resources  resource_id:str, org_id:str, service:str, region:str, created_at:str, state:str, tags_json:str
tickets    ticket_id:str, org_id:str, category:str, severity:str, created_at:str, resolved_at:str, csat:float64, sla_breached:bool
marketing  touch_id:str, org_id:str, campaign:str, channel:str, timestamp:str, clicked:bool, converted:bool
nps        org_id:str, survey_date:str, nps_score:float64, comment:str
billing    invoice_id:str, org_id:str, month:str, subtotal:float64, credits:float64, taxes:float64, currency:str, exchange_rate_to_usd:float64


## 3 · Valores nulos

In [13]:
partes = []
for k, d in dfs.items():
    n = d.isna().sum()
    n = n[n > 0]
    partes.append(pd.DataFrame({"fuente": k, "columna": n.index, "nulos": n.values,
                                "% de filas": (n.values / len(d) * 100).round(1)}))
nulos = pd.concat(partes, ignore_index=True)
nulos

,fuente,columna,nulos,% de filas
0,customers,nps_score,11,13.8
1,users,last_login,139,17.4
2,resources,tags_json,83,20.8
3,tickets,resolved_at,240,24.0
4,tickets,csat,254,25.4
5,nps,nps_score,19,20.7
6,nps,comment,10,10.9
7,billing,credits,137,57.1


## 4 · Calidad por fuente

### 4.1 · `customers_orgs`

In [24]:
c = dfs["customers"]
print("nps_score  min / max:", c.nps_score.min(), "/", c.nps_score.max(), "| nulos:", int(c.nps_score.isna().sum()))
print("  fuera de [0, 10]    :", int(((c.nps_score < 0) | (c.nps_score > 10)).sum()))
print("  fuera de [-100, 100]:", int(((c.nps_score < -100) | (c.nps_score > 100)).sum()))
print()
print("lifecycle_stage:", c.lifecycle_stage.value_counts().to_dict())
print("plan_tier      :", c.plan_tier.value_counts().to_dict())
print("hq_region      :", c.hq_region.value_counts().to_dict())
print("industrias     :", c.industry.nunique())
print()
print("is_enterprise según plan_tier (¿consistente?):")
pd.crosstab(c.plan_tier, c.is_enterprise)

nps_score  min / max: -38.0 / 101.0 | nulos: 11
  fuera de [0, 10]    : 59
  fuera de [-100, 100]: 1

lifecycle_stage: {'active': 54, 'at_risk': 11, 'churned': 6, 'prospect': 6, 'lead': 3}
plan_tier      : {'standard': 41, 'pro': 21, 'enterprise': 10, 'free': 8}
hq_region      : {'eu-central': 16, 'ap-south': 15, 'us-east': 14, 'ap-northeast': 11, 'sa-east': 10, 'us-west': 7, 'eu-west': 7}
industrias     : 10

is_enterprise según plan_tier (¿consistente?):


is_enterprise,False,True
plan_tier,,
enterprise,9,1
free,6,2
pro,15,6
standard,33,8


### 4.2 · `users`

In [15]:
u = dfs["users"].assign(created=lambda x: pd.to_datetime(x.created_at), login=lambda x: pd.to_datetime(x.last_login))
print("roles :", u.role.value_counts().to_dict())
print("active:", u.active.value_counts().to_dict())
print("created_at   rango:", u.created.min().date(), "→", u.created.max().date())
print("last_login   rango:", u.login.min().date(), "→", u.login.max().date())
print()
print("last_login nulo           :", int(u.login.isna().sum()))
print("last_login < created_at   :", int((u.login < u.created).sum()), "(inconsistente)")
print("usuarios inactivos        :", int((~u.active).sum()))
print("last_login nulo e inactivo:", int((u.login.isna() & ~u.active).sum()))

roles : {'data_engineer': 201, 'devops': 162, 'developer': 130, 'analyst': 113, 'ml_engineer': 105, 'admin': 89}
active: {True: 728, False: 72}
created_at   rango: 2025-05-04 → 2025-08-11
last_login   rango: 2025-05-14 → 2025-08-31

last_login nulo           : 139
last_login < created_at   : 232 (inconsistente)
usuarios inactivos        : 72
last_login nulo e inactivo: 12


### 4.3 · `resources`

In [16]:
r = dfs["resources"]
print("servicios:", r.service.value_counts().to_dict())
print("regiones :", r.region.value_counts().to_dict())
print("estados  :", r.state.value_counts().to_dict())
print()


def json_valido(s):
    try:
        json.loads(s)
        return True
    except (TypeError, ValueError):
        return False


con_tags = r.tags_json.dropna()
print("tags_json nulo     :", int(r.tags_json.isna().sum()))
print("tags_json inválido :", int((~con_tags.map(json_valido)).sum()))
tags = pd.Series([t for lista in con_tags.map(json.loads) for t in lista])
print("etiquetas distintas:", tags.value_counts().to_dict())

servicios: {'compute': 116, 'storage': 71, 'database': 68, 'networking': 64, 'analytics': 42, 'genai': 39}
regiones : {'ap-northeast': 72, 'us-east': 69, 'us-west': 61, 'sa-east': 57, 'ap-south': 55, 'eu-west': 46, 'eu-central': 40}
estados  : {'running': 242, 'stopped': 119, 'terminated': 39}

tags_json nulo     : 83
tags_json inválido : 0
etiquetas distintas: {'env:dev': 93, 'team:ml': 90, 'pii:true': 85, 'env:prod': 80, 'team:core': 78, 'costcenter:beta': 75, 'costcenter:alpha': 75, 'backup:daily': 67}


### 4.4 · `support_tickets`

In [17]:
t = dfs["tickets"].assign(created=lambda x: pd.to_datetime(x.created_at), resolved=lambda x: pd.to_datetime(x.resolved_at))
t["dias_resolucion"] = (t.resolved - t.created).dt.days
print("severidad :", t.severity.value_counts().to_dict())
print("categoría :", t.category.value_counts().to_dict())
print("created_at rango:", t.created.min().date(), "→", t.created.max().date())
print()
print("csat valores:", t.csat.value_counts().sort_index().to_dict(), "| nulos:", int(t.csat.isna().sum()))
print("resolved_at nulo (abiertos)              :", int(t.resolved.isna().sum()))
print("resolved_at < created_at                 :", int((t.resolved < t.created).sum()))
print("csat informado SIN resolved_at           :", int((t.csat.notna() & t.resolved.isna()).sum()), "(inconsistente)")
print("sla_breached verdadero SIN resolved_at   :", int((t.sla_breached & t.resolved.isna()).sum()), "(puede ser válido: vencido y aún abierto)")
print()
print("días de resolución:")
t.dias_resolucion.describe().round(1).to_frame().T

severidad : {'low': 412, 'medium': 328, 'high': 204, 'critical': 56}
categoría : {'integration': 181, 'billing': 180, 'availability': 165, 'usability': 161, 'performance': 158, 'security': 155}
created_at rango: 2025-05-09 → 2025-08-31

csat valores: {0.0: 11, 1.0: 45, 2.0: 127, 3.0: 242, 4.0: 197, 5.0: 95, 6.0: 28, 7.0: 1} | nulos: 254
resolved_at nulo (abiertos)              : 240
resolved_at < created_at                 : 0
csat informado SIN resolved_at           : 172 (inconsistente)
sla_breached verdadero SIN resolved_at   : 25 (puede ser válido: vencido y aún abierto)

días de resolución:


,count,mean,std,min,25%,50%,75%,max
dias_resolucion,760.0,4.3,4.7,0.0,1.0,3.0,6.0,31.0


### 4.5 · `marketing_touches` y `nps_surveys`

In [18]:
m = dfs["marketing"]
print("canales:", m.channel.value_counts().to_dict())
print("campañas:", m.campaign.value_counts().to_dict())
print("rango de fechas:", m.timestamp.min(), "→", m.timestamp.max())
print("converted sin clicked:", int((m.converted & ~m.clicked).sum()), "(posible conversión sin click directo)")
print()
n = dfs["nps"]
print("nps_surveys: filas", len(n), "| orgs distintas", n.org_id.nunique(), "| (org, fecha) repetidas", int(n.duplicated(["org_id", "survey_date"]).sum()))
print("nps_score min / max:", n.nps_score.min(), "/", n.nps_score.max(), "| nulos:", int(n.nps_score.isna().sum()), "| comment nulo:", int(n.comment.isna().sum()))
print("comentarios:", n.comment.value_counts().to_dict())

canales: {'event': 401, 'email': 385, 'ads': 358, 'in_app': 356}
campañas: {'upgrade_enterprise': 271, 'webinar_finops': 263, 'genai_launch': 252, 'welcome': 247, 'upgrade_pro': 245, 'security_week': 222}
rango de fechas: 2025-05-04 → 2025-08-31
converted sin clicked: 96 (posible conversión sin click directo)

nps_surveys: filas 92 | orgs distintas 60 | (org, fecha) repetidas 0
nps_score min / max: -16.0 / 68.0 | nulos: 19 | comment nulo: 10
comentarios: {'Missing features': 20, 'Love genAI features': 14, 'Complex billing': 13, 'Stable but slow': 13, 'Too expensive': 12, 'Great support': 10}


### 4.6 · `billing_monthly`

Hay tres monedas con su tipo de cambio a USD. El cálculo de revenue en USD depende de decisiones abiertas: qué significa `credits` vacío y cómo tratar la tasa de las facturas en USD.

In [19]:
b = dfs["billing"]
print("meses   :", sorted(b.month.unique()))
print("facturas por organización:", b.groupby("org_id").size().value_counts().to_dict(), "| orgs:", b.org_id.nunique())
print("monedas :", b.currency.value_counts().to_dict())
print()
print("tipo de cambio a USD por moneda:")
print(b.groupby("currency").exchange_rate_to_usd.agg(["min", "median", "max"]).round(5).to_string())
print()
print("subtotal por moneda (¿la magnitud es coherente con la moneda?):")
print(b.groupby("currency").subtotal.agg(["min", "median", "max"]).round(1).to_string())
print()
print("subtotal negativo :", int((b.subtotal < 0).sum()))
print("credits nulo      :", int(b.credits.isna().sum()), "| credits > 0:", int((b.credits > 0).sum()), "| credits < 0:", int((b.credits < 0).sum()))
print("taxes nulo        :", int(b.taxes.isna().sum()))

meses   : ['2025-06-01', '2025-07-01', '2025-08-01']
facturas por organización: {3: 80} | orgs: 80
monedas : {'USD': 160, 'ARS': 51, 'EUR': 29}

tipo de cambio a USD por moneda:
              min   median      max
currency                           
ARS       0.00133  0.00150  0.00162
EUR       0.99810  1.10467  1.19808
USD       0.85463  0.99515  1.11791

subtotal por moneda (¿la magnitud es coherente con la moneda?):
             min  median     max
currency                        
ARS      -1355.7   816.1  1683.7
EUR      -1095.1   697.8  2237.4
USD      -1671.8   657.6  2049.0

subtotal negativo : 13
credits nulo      : 137 | credits > 0: 102 | credits < 0: 0
taxes nulo        : 0


**Revenue exploratorio.** Hipótesis a validar (no es una regla): `neto_local = subtotal − credits + taxes`, con `credits` nulo tratado como 0, y `neto_usd = neto_local × exchange_rate_to_usd`.

In [20]:
bb = b.assign(credits0=b.credits.fillna(0))
bb["neto_local"] = bb.subtotal - bb.credits0 + bb.taxes
bb["neto_usd"] = bb.neto_local * bb.exchange_rate_to_usd
resumen = bb.groupby("currency")[["neto_local", "neto_usd"]].agg(["count", "median", "sum"]).round(1)
resumen

neto_local                  neto_usd                 
              count median       sum    count median       sum
currency                                                      
ARS              51  969.0   50292.7       51    1.4      75.7
EUR              29  844.4   28750.4       29  987.7   31965.9
USD             160  786.8  132251.4      160  809.6  132143.3

## 5 · Integridad referencial

Todas las fuentes con `org_id` deben apuntar a una organización existente en `customers_orgs`.

In [21]:
orgs = set(dfs["customers"].org_id)
filas = []
for k in ["users", "resources", "tickets", "marketing", "nps", "billing"]:
    d = dfs[k]
    filas.append({"fuente": k, "filas": len(d), "org_id nulo": int(d.org_id.isna().sum()),
                  "org_id huérfano": int((~d.org_id.isin(orgs)).sum()),
                  "orgs sin ninguna fila": len(orgs - set(d.org_id))})
pd.DataFrame(filas)

,fuente,filas,org_id nulo,org_id huérfano,orgs sin ninguna fila
0,users,800,0,0,0
1,resources,400,0,0,0
2,tickets,1000,0,0,0
3,marketing,1500,0,0,0
4,nps,92,0,0,20
5,billing,240,0,0,0
